In [41]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [42]:
df_2015 = pd.read_csv('./기부예측_자료모음/원본_복지_사회참여_문화와여가_소득과소비_노동_데이터/df_2015.csv')
df_2017 = pd.read_csv('./기부예측_자료모음/원본_복지_사회참여_문화와여가_소득과소비_노동_데이터/df_2017.csv')
df_2019 = pd.read_csv('./기부예측_자료모음/원본_복지_사회참여_문화와여가_소득과소비_노동_데이터/df_2019.csv')
df_2021 = pd.read_csv('./기부예측_자료모음/원본_복지_사회참여_문화와여가_소득과소비_노동_데이터/df_2021.csv')
df_2023 = pd.read_csv('./기부예측_자료모음/원본_복지_사회참여_문화와여가_소득과소비_노동_데이터/df_2023.csv')

In [43]:
df_2015['조사년도'] = 2015
df_2017['조사년도'] = 2017
df_2019['조사년도'] = 2019
df_2021['조사년도'] = 2021
df_2023['조사년도'] = 2023

In [44]:
df = pd.concat(
    [df_2015, df_2017, df_2019, df_2021, df_2023],
    ignore_index=True
)
df['조사년도'].isnull().sum()

np.int64(0)

In [45]:
#######################################################################


import pandas as pd

# ---------- 0) Load ----------

print("Before:", df.shape)

# ---------- 1) Drop columns with >= 50% missing ----------
threshold = 0.5  # 50%
missing_ratio = df.isna().mean()  # 각 컬럼별 결측치 비율
cols_to_drop = missing_ratio[missing_ratio >= threshold].index

df_cleaned = df.drop(columns=cols_to_drop)

print("After:", df_cleaned.shape)
print(f"🗑 Dropped {len(cols_to_drop)} columns (>=50% missing).")

# ---------- 2) Save cleaned data ----------
output_path = "df_after_drop50.csv"
df_cleaned.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Cleaned dataset saved to: {output_path}")

# (선택) 드롭된 컬럼 리스트도 저장
dropped_cols_path = "dropped_cols_over50.csv"
pd.Series(cols_to_drop, name="dropped_columns").to_csv(dropped_cols_path, index=False, encoding="utf-8-sig")

print(f"Dropped columns list saved to: {dropped_cols_path}")

########################################################################
import pandas as pd

# 데이터 불러오기
path = "df_after_drop50.csv"
df = pd.read_csv(path, low_memory=False)

# 1) 범주형 컬럼 찾기 (숫자가 아닌 dtype)
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print("범주형 컬럼 개수:", len(cat_cols))
print("범주형 컬럼 목록:\n", cat_cols)

# 2) 각 범주형 컬럼별 고유값 개수 & 결측치 비율
cat_summary = pd.DataFrame({
    "n_unique": df[cat_cols].nunique(dropna=True),
    "n_missing": df[cat_cols].isna().sum(),
    "missing_pct": (df[cat_cols].isna().mean() * 100).round(2)
}).sort_values("n_unique")

print("\n범주형 변수 요약:")
print(cat_summary)

# 3) 각 범주형 변수별 예시 값 (앞 5개)
print("\ 예시 값 확인:")
for c in cat_cols:
    examples = df[c].dropna().unique()[:5]
    print(f"- {c} (unique {df[c].nunique(dropna=True)}): {examples}")

#######################################################################

# ---------- 1) '연령' 컬럼 처리 ----------
# '만연령'은 유지하고, 나머지 '연령' 들어간 컬럼 삭제
drop_age_cols = [c for c in df.columns if ("연령" in c and c != "만연령")]
df = df.drop(columns=drop_age_cols)

print("🗑 삭제된 연령 관련 컬럼:", drop_age_cols)

# ---------- 2) 특정 코드 컬럼 원핫인코딩 ----------
onehot_cols = ["직장산업대분류코드", "직업대분류코드"]

# 실제 존재하는 컬럼만 대상으로 처리
onehot_cols = [c for c in onehot_cols if c in df.columns]

df = pd.get_dummies(df, columns=onehot_cols, drop_first=True)

print("One-Hot Encoding 완료:", onehot_cols)
print(" 최종 Shape:", df.shape)
output_path = "df_after_age_onehot.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

########################################################################
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import VarianceThreshold

# ---------- 0) Load ----------
path = "df_after_age_onehot.csv"
df = pd.read_csv(path, low_memory=False)

target_col = "기부여부"

# ---------- 1) 결측치 50% 이상 컬럼 제거 ----------
df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])

# ---------- 2) 수치형 / 범주형 분리 후 결측치 채우기 ----------
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

# 수치형 → 중앙값
imputer_num = SimpleImputer(strategy="median")  ###
df[num_cols] = imputer_num.fit_transform(df[num_cols])

# 범주형 → 최빈값 후 숫자형 변환
imputer_cat = SimpleImputer(strategy="most_frequent")
df[cat_cols] = imputer_cat.fit_transform(df[cat_cols])
for c in cat_cols:
    df[c] = df[c].astype("category").cat.codes  # 숫자형 코드로 변환

# ---------- 3) 타겟 분리 ----------
df[target_col] = df[target_col].astype(int)
X = df.drop(columns=[target_col])
y = df[target_col]

# ---------- 4) 저카디널리티(고유값 ≤ 10) 컬럼만 원핫 인코딩 ----------
low_cardinality_cols = X.columns[X.nunique() <= 10]
X = pd.get_dummies(X, columns=low_cardinality_cols, drop_first=True)

# ---------- 5) 분산이 0.01 이하인 컬럼 제거 (컬럼명 유지) ----------
selector = VarianceThreshold(threshold=0.01)
X_sel = selector.fit_transform(X)
X = pd.DataFrame(X_sel, columns=X.columns[selector.get_support(indices=True)])

# ---------- 6) 다중공선성 제거 (상관계수 > 0.9) ----------
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
drop_cols = [col for col in upper.columns if any(upper[col] > 0.9)]
X = X.drop(columns=drop_cols)

# ---------- 7) 여러 버전 생성 ----------
X_raw = X.copy()

scaler_std = StandardScaler()
X_std = pd.DataFrame(scaler_std.fit_transform(X), columns=X.columns)

scaler_mm = MinMaxScaler()
X_mm = pd.DataFrame(scaler_mm.fit_transform(X), columns=X.columns)

print("Shapes")
print("Raw:", X_raw.shape, "Std:", X_std.shape, "MinMax:", X_mm.shape, "Target:", y.shape)
# ---------- 8) 결과 저장 ----------
X_raw.to_csv("X_raw.csv", index=False, encoding="utf-8-sig")
X_std.to_csv("X_std.csv", index=False, encoding="utf-8-sig")
X_mm.to_csv("X_mm.csv", index=False, encoding="utf-8-sig")
y.to_csv("y.csv", index=False, encoding="utf-8-sig")

print("저장 완료")
print(" - X_raw.csv:", X_raw.shape)
print(" - X_std.csv:", X_std.shape)
print(" - X_mm.csv:", X_mm.shape)
print(" - y.csv:", y.shape)

########################################################################

import pandas as pd

# 지울 변수 목록
drop_cols = ["분류코드_기부여부_1"]

# 파일 리스트
files = ["X_mm.csv", "X_raw.csv", "X_std.csv"]

for f in files:
    df = pd.read_csv(f, low_memory=False)

    # 존재하는 컬럼만 제거
    cols_to_drop = [c for c in drop_cols if c in df.columns]
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        print(f"🗑 {f} 에서 {cols_to_drop} 제거 완료")
    else:
        print(f"{f} 에서 제거할 컬럼이 없음")

    # 덮어쓰기 저장
    df.to_csv(f, index=False, encoding="utf-8-sig")
    print(f"저장 완료: {f}")

print("모든 파일 처리 완료")

# -*- coding: utf-8 -*-
import pandas as pd

# 삭제할 키워드
keywords = ["미기부사유코드", "기부희망분야_1순위코드", "기부희망분야_2순위코드", "기부희망분야_3순위코드"]

# 대상 파일
files = ["X_mm.csv", "X_raw.csv", "X_std.csv"]

for f in files:
    df = pd.read_csv(f, low_memory=False)

    # 키워드 포함된 컬럼 필터링
    cols_to_drop = [c for c in df.columns if any(k in c for k in keywords)]

    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        print(f"🗑 {f} → {len(cols_to_drop)}개 컬럼 삭제")
        # 삭제된 컬럼명 일부 확인
        print("   예시:", cols_to_drop[:10])
    else:
        print(f" {f} → 삭제할 컬럼 없음")

    # 덮어쓰기 저장
    df.to_csv(f, index=False, encoding="utf-8-sig")
    print(f" 저장 완료: {f}")

print(" 모든 파일 처리 완료")

########################################################################
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np

# 모델 관련
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, KFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

# 언더샘플링
from imblearn.under_sampling import RandomUnderSampler

# 부스터 모델
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 딥러닝 (CNN)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# -------------------------------
# 0. 데이터 불러오기
X = pd.read_csv("X_mm.csv")  ##### 파일 원하는걸루 바꾸세요.
y = pd.read_csv("y.csv").squeeze()

# y 값 확인 및 변환 (혹시 모를 [1,2] → [0,1])
if set(y.unique()) == {1, 2}:
    y = y.map({1: 0, 2: 1}).astype(int)

print(" 데이터 로드 완료:", X.shape, y.shape, "Unique y:", y.unique())


# -------------------------------
# 1. 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba, fold):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)
    return {
        "Fold": fold,
        "Algorithm": name,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    }


# -------------------------------
# 2. 파라미터 후보 (최소화)
param_grids = {
    "Logistic Regression": {'C': [0.1, 1, 10]},
    "Random Forest": {'n_estimators': [100, 200], 'max_depth': [5, 10]},
    "XGBoost": {'n_estimators': [100], 'max_depth': [3, 5]},
    "LightGBM": {'n_estimators': [100], 'num_leaves': [31, 50]},
    "CatBoost": {'depth': [4, 6], 'iterations': [100], 'learning_rate': [0.1]},
    "MLP": {'hidden_layer_sizes': [(64,), (64, 32)], 'alpha': [0.0001, 0.001]}
}

# -------------------------------
# 3. 기본 모델
base_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
    "MLP": MLPClassifier(max_iter=300, random_state=42)
}

# -------------------------------
# 4. 학습 + 평가 (5-Fold, 메모리 최적화)
results = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    # 스케일링
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
    X_valid_scaled = scaler.transform(X_valid).astype(np.float32)

    # 언더샘플링
    rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
    X_res, y_res = rus.fit_resample(X_train_scaled, y_train)

    # 메모리 절약을 위해 float32 변환
    X_res = X_res.astype(np.float32)

    # 전통 ML 모델 (RandomizedSearchCV + n_jobs=2)
    for name, model in base_models.items():
        clf = RandomizedSearchCV(
            model,
            param_distributions=param_grids[name],
            n_iter=2, cv=3, scoring='f1',
            n_jobs=2, random_state=42
        )
        clf.fit(X_res, y_res)
        best_model = clf.best_estimator_
        y_pred = best_model.predict(X_valid_scaled)
        y_proba = best_model.predict_proba(X_valid_scaled)[:, 1]
        results.append(evaluate_model(name, y_valid, y_pred, y_proba, fold))

    # CNN
    cnn = Sequential([
        Dense(64, activation='relu', input_shape=(X_res.shape[1],)),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    cnn.fit(X_res, y_res, epochs=5, batch_size=32, verbose=0)  # epochs 줄임

    y_pred_cnn = cnn.predict(X_valid_scaled).flatten()
    y_pred_cnn_label = (y_pred_cnn >= 0.5).astype(int)
    results.append(evaluate_model("CNN", y_valid, y_pred_cnn_label, y_pred_cnn, fold))

# -------------------------------
# 5. KNN 평가 (별도)
X_scaled_all = MinMaxScaler().fit_transform(X).astype(np.float32)
X_res_all, y_res_all = RandomUnderSampler(sampling_strategy=0.5, random_state=42).fit_resample(X_scaled_all, y)

knn = KNeighborsClassifier(n_neighbors=5)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred_knn = cross_val_predict(knn, X_res_all, y_res_all, cv=kf, method='predict')
y_proba_knn = cross_val_predict(knn, X_res_all, y_res_all, cv=kf, method='predict_proba')[:, 1]
results.append(evaluate_model("KNN", y_res_all, y_pred_knn, y_proba_knn, fold='All'))

# -------------------------------
# 6. 결과 정리 + 저장
result_df = pd.DataFrame(results)

summary = result_df.groupby("Algorithm")[[
    "Precision", "Recall", "Specificity", "F1-score",
    "Accuracy", "Balanced Accuracy", "AUC"
]].mean().round(4).sort_values("AUC", ascending=False)

result_df.to_csv("all_fold_results.csv", index=False, encoding="utf-8-sig")
summary.to_csv("model_summary.csv", encoding="utf-8-sig")

print("저장 완료")
print(" - all_fold_results.csv : Fold별 전체 성능 기록")
print(" - model_summary.csv : 모델별 평균 성능 요약")

Before: (184915, 465)
After: (184915, 150)
🗑 Dropped 315 columns (>=50% missing).
Cleaned dataset saved to: df_after_drop50.csv
Dropped columns list saved to: dropped_cols_over50.csv
범주형 컬럼 개수: 14
범주형 컬럼 목록:
 ['직장산업대분류', '직업대분류', '분류코드_연령성별8', '분류코드_연령성별9', '분류코드_연령교육정도별1', '분류코드_장애인복지카드_소유여부', '분류코드_기부여부', '분류코드_현금기부여부', '분류코드_물품기부여부', '분류코드_자원봉사여부', '분류코드_계층의식', '분류코드_주관적소득수준', '분류코드_소득만족도', '분류코드_고용안정성여부']

범주형 변수 요약:
                   n_unique  n_missing  missing_pct
분류코드_기부여부                 2          0         0.00
분류코드_현금기부여부               2          0         0.00
분류코드_장애인복지카드_소유여부         2          0         0.00
분류코드_물품기부여부               2          0         0.00
분류코드_자원봉사여부               2          0         0.00
분류코드_고용안정성여부              2      79050        42.75
분류코드_계층의식                 3          0         0.00
분류코드_주관적소득수준              3      92385        49.96
분류코드_소득만족도                3      37463        20.26
분류코드_연령성별9               14      89197        48.24
분류코

ValueError: "sampling_strategy" can be a float only when the type of target is binary. For multi-class, use a dict.

In [7]:
target_col = '기부여부'

In [8]:
# 기부:1, 미기부:0
df['기부여부'] = df['기부여부'].replace({2:0})

In [9]:
# unique 값이 1개인 컬럼 찾기
one_unique_cols = [col for col in df.columns if df[col].nunique() == 1]
print("unique가 1개인 컬럼:", one_unique_cols)
df = df.drop(columns=one_unique_cols)

unique가 1개인 컬럼: ['분류코드_만13세이상여부', '분류코드_만15세이상여부', '분류코드_점유형태2', '분류코드_주택형태2', '분류코드_연령2', '분류코드_가구원수2', '분류코드_연령15', '분류코드_연령16']


In [10]:
drop_cols_text = '''
기부내용_기타_물품횟수
기부내용_기타_현금금액
기부내용_기타_현금주기
기부내용_기타_현금횟수
기부내용_대상자직접후원_물품횟수
기부내용_대상자직접후원_현금금액
기부내용_대상자직접후원_현금주기
기부내용_대상자직접후원_현금횟수
기부내용_모금단체후원_현금금액
기부내용_모금단체후원_현금주기
기부내용_모금단체후원_현금횟수
기부내용_물품주기
기부내용_물품횟수
기부내용_물품후원단체후원_물품횟수
기부내용_언론기관후원_물품횟수
기부내용_언론기관후원_현금금액
기부내용_언론기관후원_현금주기
기부내용_언론기관후원_현금횟수
기부내용_종교단체후원_물품횟수
기부내용_종교단체후원_현금금액
기부내용_종교단체후원_현금주기
기부내용_종교단체후원_현금횟수
기부내용_직장기업후원_물품횟수
기부내용_직장기업후원_현금금액
기부내용_직장기업후원_현금주기
기부내용_직장기업후원_현금횟수
기부대상인지경로
기부사유
분류코드_기부여부
분류코드_물품기부여부
분류코드_물품기부정기성여부
분류코드_현금기부여부
분류코드_현금기부정기성여부
정기기부_물품기부_주기
정기기부_물품기부주기
정기기부_물품기부여부
정기기부_현금기부_주기
정기기부_현금기부여부
정기기부_현금기부주기
분류코드_연령성별1
분류코드_연령성별10
분류코드_연령성별11
분류코드_연령성별2
분류코드_연령성별3
분류코드_연령성별4
분류코드_연령성별5
분류코드_연령성별6
분류코드_연령성별7
분류코드_연령성별8
분류코드_연령성별9
가구일련번호
가구가중값
가구원가중값
가구번호
표본층번호
승수(가구)_weight
승수(가구원)_weight
분류코드_직업대분류
분류코드_직업별
분류코드_직장산업대분류
'''
drop_cols_original = drop_cols_text.split()
df = df.drop(columns=[col for col in drop_cols_original if col in df.columns])

In [11]:
drop_cols_text = '''
기부종류
기부내용_기타_물품횟수
기부내용_기타_현금금액
기부내용_기타_현금주기
기부내용_기타_현금횟수
기부내용_대상자직접후원_물품횟수
기부내용_대상자직접후원_현금금액
기부내용_대상자직접후원_현금주기
기부내용_대상자직접후원_현금횟수
기부내용_모금단체후원_현금금액
기부내용_모금단체후원_현금주기
기부내용_모금단체후원_현금횟수
기부내용_물품주기
기부내용_물품횟수
기부내용_물품후원단체후원_물품횟수
기부내용_언론기관후원_물품횟수
기부내용_언론기관후원_현금금액
기부내용_언론기관후원_현금주기
기부내용_언론기관후원_현금횟수
기부내용_종교단체후원_물품횟수
기부내용_종교단체후원_현금금액
기부내용_종교단체후원_현금주기
기부내용_종교단체후원_현금횟수
기부내용_직장기업후원_물품횟수
기부내용_직장기업후원_현금금액
기부내용_직장기업후원_현금주기
기부내용_직장기업후원_현금횟수
기부대상인지경로
기부사유
미기부사유
분류코드_기부여부
분류코드_물품기부여부
분류코드_물품기부정기성여부
분류코드_현금기부여부
분류코드_현금기부정기성여부
정기기부_물품기부_주기
정기기부_물품기부주기
정기기부_물품기부여부
정기기부_현금기부_주기
정기기부_현금기부여부
정기기부_현금기부주기
분류코드_연령성별1
분류코드_연령성별10
분류코드_연령성별11
분류코드_연령성별2
분류코드_연령성별3
분류코드_연령성별4
분류코드_연령성별5
분류코드_연령성별6
분류코드_연령성별7
분류코드_연령성별8
분류코드_연령성별9
가구일련번호
가구가중값
가구원가중값
가구번호
승수(가구)_weight
승수(가구원)_weight
분류코드_직업대분류
분류코드_직업별
분류코드_직장산업대분류
'''
drop_cols_original = drop_cols_text.split()
df = df.drop(columns=[col for col in drop_cols_original if col in df.columns])

In [12]:
value_continued = '''
만연령
사회적관계망_가사도움요청대상인원수
사회적관계망_자금차입도움요청대상인원수
사회적관계망_대화상대도움요청대상인원수
자원봉사_아동청소년노인장애인재소자관련횟수
자원봉사_아동청소년노인장애인재소자관련시간수
자원봉사활동_환경보전범죄예방관련횟수
자원봉사활동_환경보전범죄예방관련시간수
자원봉사활동_자녀교육관련횟수
자원봉사활동_자녀교육관련시간수
자원봉사활동_국가지역행사관련횟수
자원봉사활동_국가지역행사관련시간수
자원봉사활동_재해지역주민돕기시설복구관련횟수
자원봉사활동_재해지역주민돕기시설복구관련시간수
자원봉사활동_기타일반봉사관련횟수
자원봉사활동_기타일반봉사관련시간수
독서_잡지류권수
독서_교양서적권수
독서_직업직무관련서적권수
독서_생활취미정보서적권수
독서_기타권수
문화예술스포츠관람_음악연주회횟수
문화예술스포츠관람_연극마당극뮤지컬횟수
문화예술스포츠관람_무용횟수
문화예술스포츠관람_영화횟수
문화예술스포츠관람_박물관횟수
문화예술스포츠관람_미술관횟수
문화예술스포츠관람_스포츠횟수
레저시설_관광명소유적지국립공원이용횟수
레저시설_온천장스파이용횟수
레저시설_골프장이용횟수
레저시설_스키장이용횟수
레저시설_해수욕장이용횟수
레저시설_산림욕장휴양림이용횟수
레저시설_놀이공원이용횟수
레저시설_수영장워터파크포함이용횟수
레저시설_기타이용횟수
국내관광여행_숙박여행횟수
국내관광여행_숙박여행1회평균숙박일수
국내관광여행_당일여행횟수
해외여행경험_관광횟수
해외여행경험_가족친지관련횟수
해외여행경험_업무횟수
해외여행경험_교육횟수
'''

numeric_cols_original = value_continued.split()
# df = df.drop(columns=[col for col in drop_cols if col in df.columns])

In [13]:
value_nominal = '''
미기부사유
유산기부의사여부
가구주와의관계
거처의종류
기부문화확산_1순위
기부문화확산_2순위
기부문화확산_3순위
기부문화확산1순위
기부문화확산2순위
기부문화확산3순위
기부여부
기부희망분야1순위
기부희망분야2순위
기부희망분야3순위
기부종류
기부하지않은이유
기부한이유
유산기부의사여부
기부희망분야_1순위
기부희망분야_2순위
기부희망분야_3순위
긴축소비지출항목_1순위
긴축소비지출항목_2순위
긴축소비지출항목_3순위
노후준비방법_노후미준비사유
노후준비방법_부수적인것
노후준비방법_부수적방법
노후를보내고싶은방법
노후를위한사회적관심사
노후를위한사회적역할
노후생활주요활동_1순위
노후생활주요활동_2순위
노후생활주요활동_3순위
노후준비방법_주된방법
노후희망활동_1순위
노후희망활동_2순위
노후희망활동_3순위
노후희망활동
분류코드맞벌이여부
분류코드맞벌이여부2
분류코드산업별
분류코드_맞벌이1여부
분류코드_맞벌이2여부
분류코드_산업별
분류코드_점유형태
분류코드_점유형태1
분류코드_점유형태2
분류코드_종사자지위
분류코드_주택형태
분류코드_주택형태1
분류코드_주택형태2
분류코드_청소년교육정도코드1
분류코드_청소년교육정도코드2
분류코드_혼인상태
분류코드_혼인상태2
산업
생활비마련방법
생활비마련방법_본인및배우자부담
선호장례방법
신문인터넷신문_보는분야
신문일반신문_보는분야
앞으로하고싶은여가활동1순위
앞으로하고싶은여가활동2순위
앞으로하고싶은여가활동3순위
여가활용_불만족사유
여가활동_동반자
여성취업장애요인_1순위
여성취업장애요인_2순위
여성취업장애요인_3순위
여성취업에대한견해
우선실시장애인복지사업_1순위
우선실시장애인복지사업_2순위
우선실시장애인복지사업_3순위
원격수업이효과적이지않은이유1순위
원격수업이효과적이지않은이유2순위
원격수업이효과적이지않은이유3순위
자원봉사활동인지경로
장애인과의유대관계여부및대상
장애인과의유대관계여부및대상대상1순위
장애인과의유대관계여부및대상대상2순위
장애인과의유대관계여부및대상대상3순위
장애인유대관계대상_1순위
장애인유대관계대상_2순위
장애인유대관계대상_3순위
재택근무가비효율적인이유1순위
재택근무가비효율적인이유2순위
재택근무가비효율적인이유3순위
재학상태
전문성을활용한자원봉사활동
전문성활용자원봉사활동내용
점유형태
종사상의지위
주말여가활동_동반자
주말휴일여가활용_1순위
주말휴일여가활용_2순위
주말휴일여가활용_3순위
주중여가활동_동반자
주중여가활용_1순위
주중여가활용_2순위
주중여가활용_3순위
직업선택요인_1순위
직업선택요인_2순위
직업선택요인_3순위
직업대분류
직업
직장산업대분류
참여단체_1순위
참여단체_2순위
참여단체_3순위
청년이선호하는직장
향후추가대상공공시설_3순위
행정구역(시도)
향후추가대상공공시설_1순위
향후추가대상공공시설_2순위
향후추가대상복지서비스_1순위
향후추가대상복지서비스_2순위
향후추가대상복지서비스_3순위
향후희망거처
향후희망여가활동_1순위
향후희망여가활동_2순위
향후희망여가활동_3순위
현재자녀동거사유_1순위
현재자녀동거사유_2순위
현재자녀동거사유_3순위
현재자녀비동거사유_1순위
현재자녀비동거사유_2순위
현재자녀비동거사유_3순위
혼인상태
'''

nominal_cols_original = value_nominal.split()

In [14]:
ordinal_cols_original = [col for col in df.columns
                if col not in numeric_cols_original and col not in nominal_cols_original and col != target_col]
print(len(ordinal_cols_original))

287


In [15]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [18]:
# 결측치 비율이 50% 이상인 컬럼 제거
missing_ratio = df.isnull().sum() / len(df)
cols_to_drop = missing_ratio[missing_ratio >= 0.5].index.tolist()
df = df.drop(columns=cols_to_drop)

# 컬럼 리스트 업데이트 (제거된 컬럼들 반영)
numeric_cols = [col for col in numeric_cols_original if col in df.columns]
nominal_cols = [col for col in nominal_cols_original if col in df.columns]
ordinal_cols = [col for col in ordinal_cols_original if col in df.columns]

print(f"전처리 후 데이터 형태: {df.shape}")

50% 이상 결측치를 가진 컬럼들: ['재학학년', '사회보험료부담인식_건강보험부담정도', '사회보험료부담인식_국민연금부담정도', '사회보험료부담인식_고용보험부담정도', '장애인견해', '노후준비방법_부수적방법', '노후준비방법_노후미준비사유', '생활비마련방법', '생활비마련방법_본인및배우자부담', '생활비마련방법_지원자녀및친척동거여부', '현재자녀동거여부', '현재자녀동거사유_1순위', '현재자녀동거사유_2순위', '현재자녀동거사유_3순위', '현재자녀비동거사유_1순위', '현재자녀비동거사유_2순위', '현재자녀비동거사유_3순위', '향후자녀동거의향여부', '향후희망거처', '사회적관계망_자금차입도움요청대상인원수', '단체참여여부', '참여단체_1순위', '참여단체_2순위', '참여단체_3순위', '기부희망분야_1순위', '기부희망분야_2순위', '기부희망분야_3순위', '자원봉사활동_아동청소년노인장애인재소자관련횟수', '자원봉사활동_아동청소년노인장애인재소자관련시간수', '자원봉사활동_환경보전범죄예방관련횟수', '자원봉사활동_환경보전범죄예방관련시간수', '자원봉사활동_자녀교육관련횟수', '자원봉사활동_자녀교육관련시간수', '자원봉사활동_국가및지역행사관련횟수', '자원봉사활동_국가및지역행사관련시간수', '자원봉사활동_재해지역주민돕기및시설복구관련횟수', '자원봉사활동_재해지역주민돕기및시설복구관련시간수', '자원봉사활동_기타일반봉사관련횟수', '자원봉사활동_기타일반봉사관련시간수', '자원봉사활동정기성여부', '자원봉사활동정기성주기', '자원봉사활동인지경로', '전문성활용자원봉사활동_참석여부', '전문성활용자원봉사활동_내용', '독서_잡지류권수', '독서_교양서적권수', '독서_직업직무관련서적권수', '독서_생활취미정보서적권수', '독서_기타권수', '문화예술스포츠관람_음악연주회콘서트횟수', '문화예술스포츠관람_연극마당극뮤지컬횟수', '문화예술스포츠관람_무용횟수', '문화예술스포츠관람_영화횟수', '문화예술스포츠관람_박물관횟수', '문화예술스포츠관람_미술관횟수',

In [19]:
# ---------- 1) 결측치 50% 이상 컬럼 제거 ----------
df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])

# ---------- 2) 수치형 / 범주형 분리 후 결측치 채우기 ----------
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

# 수치형 → 중앙값
imputer_num = SimpleImputer(strategy="median")
df[num_cols] = imputer_num.fit_transform(df[num_cols])

# 범주형 → 최빈값 후 숫자형 변환
imputer_cat = SimpleImputer(strategy="most_frequent")
df[cat_cols] = imputer_cat.fit_transform(df[cat_cols])
for c in cat_cols:
    df[c] = df[c].astype("category").cat.codes  # 숫자형 코드로 변환

In [20]:
# 데이터 복사본 생성
df_encoded = df.copy()

# Nominal 컬럼: 더미변수화 (숫자+문자 혼합 처리)
nominal_cols.remove('기부여부')
nominal_dummies = []
for col in nominal_cols:
    if col in df_encoded.columns:
        # 숫자와 문자가 섞인 경우를 대비해 모두 문자열로 변환
        df_encoded[col] = df_encoded[col].astype(str)

        # 더미변수 생성
        dummies = pd.get_dummies(df_encoded[col], prefix=col, drop_first=True)
        nominal_dummies.append(dummies)

        # 원본 컬럼 제거
        df_encoded = df_encoded.drop(columns=[col])

        print(f"{col}: 더미변수 {len(dummies.columns)}개 생성")

# 더미변수들을 원본 데이터프레임에 결합
if nominal_dummies:
    df_encoded = pd.concat([df_encoded] + nominal_dummies, axis=1)

# Ordinal 컬럼: 라벨 인코딩
label_encoders = {}
for col in ordinal_cols:
    if col in df_encoded.columns:
        # 숫자와 문자가 섞인 경우를 대비해 문자열로 변환 후 인코딩
        df_encoded[col] = df_encoded[col].astype(str)

        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col])
        label_encoders[col] = le

        print(f"{col}: 라벨 인코딩 완료 ({len(le.classes_)}개 클래스)")

print(f"인코딩 후 데이터 형태: {df_encoded.shape}")

유산기부의사여부: 더미변수 1개 생성
기부문화확산_1순위: 더미변수 6개 생성
기부문화확산_2순위: 더미변수 6개 생성
기부문화확산_3순위: 더미변수 6개 생성
긴축소비지출항목_1순위: 더미변수 10개 생성
노후준비방법_주된방법: 더미변수 8개 생성
노후희망활동: 더미변수 7개 생성
분류코드_맞벌이1여부: 더미변수 1개 생성
분류코드_맞벌이2여부: 더미변수 3개 생성
분류코드_산업별: 더미변수 2개 생성
분류코드_점유형태1: 더미변수 4개 생성
분류코드_종사자지위: 더미변수 3개 생성
분류코드_주택형태1: 더미변수 3개 생성
분류코드_혼인상태: 더미변수 3개 생성
선호장례방법: 더미변수 4개 생성
여성취업장애요인_1순위: 더미변수 9개 생성
여성취업장애요인_2순위: 더미변수 8개 생성
여성취업장애요인_3순위: 더미변수 8개 생성
우선실시장애인복지사업_1순위: 더미변수 9개 생성
우선실시장애인복지사업_2순위: 더미변수 9개 생성
우선실시장애인복지사업_3순위: 더미변수 9개 생성
재학상태: 더미변수 4개 생성
점유형태: 더미변수 4개 생성
주말여가활동_동반자: 더미변수 5개 생성
주말휴일여가활용_1순위: 더미변수 10개 생성
주말휴일여가활용_2순위: 더미변수 10개 생성
주말휴일여가활용_3순위: 더미변수 10개 생성
주중여가활동_동반자: 더미변수 5개 생성
주중여가활용_1순위: 더미변수 10개 생성
주중여가활용_2순위: 더미변수 10개 생성
직업선택요인_1순위: 더미변수 8개 생성
직업선택요인_2순위: 더미변수 8개 생성
직업선택요인_3순위: 더미변수 8개 생성
직업대분류: 더미변수 18개 생성
직장산업대분류: 더미변수 39개 생성
향후추가대상공공시설_3순위: 더미변수 8개 생성
향후추가대상공공시설_1순위: 더미변수 8개 생성
향후추가대상공공시설_2순위: 더미변수 8개 생성
향후추가대상복지서비스_1순위: 더미변수 8개 생성
향후추가대상복지서비스_2순위: 더미변수 8개 생성
향후추가대상복지서비스_3순위: 더미변수 8개 생성
향후희망여가활동_1순위: 더미변수 10개 

In [21]:
# df_encoded: 이미 모든 더미 컬럼과 라벨 인코딩 컬럼 포함
# nominal_cols: 원래 더미화한 컬럼 리스트
# ordinal_cols: 원래 라벨인코딩한 컬럼 리스트

# 1️⃣ 더미 변수 추출
dummy_columns = []
for col in nominal_cols:
    # df_encoded 컬럼 중 원래 컬럼명으로 시작하는 것만 선택
    dummy_columns += [c for c in df_encoded.columns if c.startswith(col + "_")]

print("더미 변수 컬럼:")
print(dummy_columns)

# 2️⃣ 라벨인코딩 컬럼 추출
label_columns = [c for c in df_encoded.columns if c in ordinal_cols]
print("라벨인코딩 컬럼:")
print(label_columns)


더미 변수 컬럼:
['유산기부의사여부_2.0', '기부문화확산_1순위_2.0', '기부문화확산_1순위_3.0', '기부문화확산_1순위_4.0', '기부문화확산_1순위_5.0', '기부문화확산_1순위_6.0', '기부문화확산_1순위_7.0', '기부문화확산_2순위_2.0', '기부문화확산_2순위_3.0', '기부문화확산_2순위_4.0', '기부문화확산_2순위_5.0', '기부문화확산_2순위_6.0', '기부문화확산_2순위_7.0', '기부문화확산_3순위_2.0', '기부문화확산_3순위_3.0', '기부문화확산_3순위_4.0', '기부문화확산_3순위_5.0', '기부문화확산_3순위_6.0', '기부문화확산_3순위_7.0', '유산기부의사여부_2.0', '긴축소비지출항목_1순위_10.0', '긴축소비지출항목_1순위_11.0', '긴축소비지출항목_1순위_2.0', '긴축소비지출항목_1순위_3.0', '긴축소비지출항목_1순위_4.0', '긴축소비지출항목_1순위_5.0', '긴축소비지출항목_1순위_6.0', '긴축소비지출항목_1순위_7.0', '긴축소비지출항목_1순위_8.0', '긴축소비지출항목_1순위_9.0', '노후준비방법_주된방법_2.0', '노후준비방법_주된방법_3.0', '노후준비방법_주된방법_4.0', '노후준비방법_주된방법_5.0', '노후준비방법_주된방법_6.0', '노후준비방법_주된방법_7.0', '노후준비방법_주된방법_8.0', '노후준비방법_주된방법_9.0', '노후희망활동_2.0', '노후희망활동_3.0', '노후희망활동_4.0', '노후희망활동_5.0', '노후희망활동_6.0', '노후희망활동_7.0', '노후희망활동_8.0', '분류코드_맞벌이1여부_220.0', '분류코드_맞벌이2여부_212.0', '분류코드_맞벌이2여부_213.0', '분류코드_맞벌이2여부_214.0', '분류코드_산업별_320.0', '분류코드_산업별_330.0', '분류코드_점유형태1_32.0', '분류코드_점유형태1_33.0', '분류코드_점유형태1_34.0', '분류코

In [22]:
# 특성과 타겟 분리
X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col]

# train/validation/test 분할 (60:20:20)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp  # 0.25 * 0.8 = 0.2
)

print(f"Train 데이터: {X_train.shape}")
print(f"Validation 데이터: {X_valid.shape}")
print(f"Test 데이터: {X_test.shape}")
print(f"타겟 분포 - Train: {y_train.value_counts().to_dict()}")
print(f"타겟 분포 - Valid: {y_valid.value_counts().to_dict()}")
print(f"타겟 분포 - Test: {y_test.value_counts().to_dict()}")

Train 데이터: (110949, 443)
Validation 데이터: (36983, 443)
Test 데이터: (36983, 443)
타겟 분포 - Train: {0.0: 83333, 1.0: 27616}
타겟 분포 - Valid: {0.0: 27778, 1.0: 9205}
타겟 분포 - Test: {0.0: 27778, 1.0: 9205}


In [23]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from imblearn.under_sampling import RandomUnderSampler

In [24]:
# 각종 모델 불러오기
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline

In [25]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.base import BaseEstimator, ClassifierMixin

In [26]:
#for scale_pos_weight for xgb
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = float(neg / pos) if pos > 0 else 1.0

In [27]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(max_depth=15, n_estimators=200, random_state=42, class_weight='balanced',
                                            min_samples_split=5,min_samples_leaf=4,bootstrap=True),
    "XGBoost": xgb.XGBClassifier(max_depth=4, learning_rate=0.1, n_estimators=100,
             subsample=0.8, colsample_bytree=0.8, min_child_weight=5, n_jobs=-1, scale_pos_weight=scale_pos_weight,
                                max_bin=255, tree_method='hist', eval_metric='logloss', random_state=42),
    "LightGBM": lgb.LGBMClassifier(max_depth=2, learning_rate=0.1, n_estimators=50, num_leaves=7, subsample=0.8,
                                colsample_bytree=0.8, min_child_samples=10, random_state=42, n_jobs=2, scale_pos_weight=scale_pos_weight,
                               class_weight='balanced', max_bin=255, force_col_wise=True, boosting_type='gbdt' ),
    "CatBoost": CatBoostClassifier(depth=2, learning_rate=0.15, iterations=50, verbose=0, random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(32,), max_iter=150, learning_rate_init=0.01, random_state=42, solver='adam',
                                 early_stopping=True, activation='relu', alpha=1e-3
    ),
}

In [28]:
# 1. PyTorch 1D CNN 모델 정의
class CNN1D(nn.Module):
    def __init__(self, input_channels, input_length):
        super(CNN1D, self).__init__()
        self.conv1 = nn.Conv1d(input_channels, 32, kernel_size=3)
        self.pool = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3)

        conv_output_size = ((input_length - 2) // 2 - 2) // 2 * 64
        self.fc1 = nn.Linear(conv_output_size, 64)
        self.dropout = nn.Dropout(0.3)  # 드롭아웃 줄임 (0.5 → 0.3)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.sigmoid(self.fc2(x))
        return x

# 래퍼 클래스에서 weight decay만 추가
class CNNClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, input_channels=1, input_length=179, epochs=15, batch_size=64, lr=0.0005, device='cpu'):
        self.input_channels = input_channels
        self.input_length = input_length
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.device = device

        self.model = CNN1D(input_channels, input_length).to(self.device)
        self.criterion = nn.BCELoss()
        # weight_decay 추가로 정규화
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=1e-4)

    def fit(self, X, y):
        if hasattr(y, "to_numpy"):
          y = y.to_numpy()

        X = torch.tensor(X, dtype=torch.float32).to(self.device)
        y = torch.tensor(y, dtype=torch.float32).unsqueeze(1).to(self.device)

        dataset = TensorDataset(X, y)
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        self.model.train()
        for epoch in range(self.epochs):
            for batch_X, batch_y in loader:
                self.optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = self.criterion(outputs, batch_y)
                loss.backward()
                self.optimizer.step()
        return self

    def predict_proba(self, X):
        self.model.eval()
        X = torch.tensor(X, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            outputs = self.model(X).cpu().numpy()
        return np.hstack([1 - outputs, outputs])  # 클래스 0,1 확률

    def predict(self, X):
        proba = self.predict_proba(X)[:, 1]
        return (proba >= 0.5).astype(int)


In [29]:
def evaluate_model(y_true, y_pred, y_prob, algorithm_name, dataset_name="Test"):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    balanced_accuracy = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_prob)

    return {
        'Dataset': dataset_name,
        'Algorithm': algorithm_name,
        'N': len(y_true),
        'True Positive': tp,
        'True Negative': tn,
        'False Positive': fp,
        'False Negative': fn,
        'Precision': precision,
        'Recall': recall,
        'Specificity': specificity,
        'F1-score': f1,
        'Accuracy': accuracy,
        'Balanced Accuracy': balanced_accuracy,
        'AUC': auc
    }

In [30]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
import numpy as np
import pandas as pd

n_splits = 3
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
THRESHOLD = 0.3
results = []

## cuda 설정
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("현재 사용되고 있는 device: ", device)

# 3. 데이터 차원 변환 (PyTorch CNN 1D 입력은 [배치, 채널, 길이])
X_train_cnn = X_train.values.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_cnn = X_test.values.reshape((X_test.shape[0], 1, X_test.shape[1]))

cnn_model = CNNClassifier(input_channels=1, input_length=X_train.shape[1], epochs=20, batch_size=32, device=device)
models['1D CNN PyTorch'] =  cnn_model

## 모델별 학습
for name, model in models.items():
    fold_metrics = []
    print(f"model name: {name}")

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        # Use .iloc for pandas DataFrames with integer indices
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # StandardScaler 적용
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_val_scaled = scaler.transform(X_val)

        # DataFrame 형태로 변환 (컬럼명 유지)
        X_tr_scaled = pd.DataFrame(X_tr_scaled, columns=X_tr.columns, index=X_tr.index)
        X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns, index=X_val.index)

        if name == '1D CNN PyTorch':
            # Convert to numpy arrays and reshape for CNN (스케일링된 데이터 사용)
            X_tr_cnn = X_tr_scaled.values.reshape((X_tr_scaled.shape[0], 1, X_tr_scaled.shape[1]))
            X_val_cnn = X_val_scaled.values.reshape((X_val_scaled.shape[0], 1, X_val_scaled.shape[1]))

            # Create fresh CNN instance for each fold
            cnn_model = CNNClassifier(
                input_channels=1,
                input_length=X_train.shape[1],
                epochs=20,
                batch_size=64,
                lr=0.0005,
                device=device
            )
            cnn_model.fit(X_tr_cnn, y_tr.values)
            y_pred = cnn_model.predict(X_val_cnn)
            y_prob = cnn_model.predict_proba(X_val_cnn)[:, 1]
            y_pred = (y_prob >= THRESHOLD).astype(int)

            # Clear GPU memory if using CUDA
            if device == 'cuda':
                torch.cuda.empty_cache()

        else:
            clf = clone(model)
            # 스케일링된 데이터로 학습
            clf.fit(X_tr_scaled, y_tr)

            # Handle probability prediction (스케일링된 데이터로 예측)
            if hasattr(clf, 'predict_proba'):
                y_prob = clf.predict_proba(X_val_scaled)[:, 1]
                y_pred = (y_prob >= THRESHOLD).astype(int)
            else:
                # Fallback to decision function for models like SVM
                y_prob = clf.decision_function(X_val_scaled)
                if y_prob.max() != y_prob.min():
                    y_prob = (y_prob - y_prob.min()) / (y_prob.max() - y_prob.min())
                else:
                    y_prob = np.full_like(y_prob, 0.5)
                # 커스텀 threshold로 예측 변환
                y_pred = (y_prob >= THRESHOLD).astype(int)

        fold_result = evaluate_model(y_val, y_pred, y_prob,
                                   algorithm_name=name,
                                   dataset_name=f"Fold {fold+1}")
        fold_metrics.append(fold_result)

    # Calculate average metrics across folds
    avg_result = {}
    numeric_keys = ['N', 'True Positive', 'True Negative', 'False Positive', 'False Negative',
                    'Precision', 'Recall', 'Specificity', 'F1-score', 'Accuracy',
                    'Balanced Accuracy', 'AUC']

    for key in numeric_keys:
        avg_result[key] = np.mean([res[key] for res in fold_metrics])

    avg_result['Algorithm'] = name
    results.append(avg_result)

현재 사용되고 있는 device:  cuda
model name: Logistic Regression
model name: Random Forest
model name: XGBoost
model name: LightGBM
[LightGBM] [Info] Number of positive: 18411, number of negative: 55555
[LightGBM] [Info] Total Bins 1691
[LightGBM] [Info] Number of data points in the train set: 73966, number of used features: 426
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warnin

In [31]:
df_fold_results = pd.DataFrame(results).round(6)
display(df_fold_results)

,N,True Positive,True Negative,False Positive,False Negative,Precision,Recall,Specificity,F1-score,Accuracy,Balanced Accuracy,AUC,Algorithm
0,36983.0,7337.666667,22788.333333,4989.333333,1867.666667,0.595263,0.797110,0.820383,0.681554,0.814590,0.808747,0.881131,Logistic Regression
1,36983.0,8510.666667,17182.000000,10595.666667,694.666667,0.445437,0.924536,0.618555,0.601213,0.694716,0.771545,0.876162,Random Forest
2,36983.0,8295.666667,19317.000000,8460.666667,909.666667,0.495081,0.901180,0.695415,0.639072,0.746631,0.798298,0.883329,XGBoost
3,36983.0,9034.333333,7924.333333,19853.333333,171.000000,0.312779,0.981424,0.285277,0.474361,0.458553,0.633350,0.874904,LightGBM
4,36983.0,7604.333333,22018.000000,5759.666667,1601.000000,0.569015,0.826079,0.792651,0.673863,0.800972,0.809365,0.878150,CatBoost
5,36983.0,6948.333333,23093.333333,4684.333333,2257.000000,0.597716,0.754817,0.831364,0.666875,0.812310,0.793090,0.871939,MLP
6,36983.0,7311.666667,22558.333333,5219.333333,1893.666667,0.583911,0.794285,0.812103,0.672789,0.807668,0.803194,0.875552,1D CNN PyTorch


In [32]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
import numpy as np
import pandas as pd
# torch 관련 import는 이 코드 블록에서 생략했습니다. 원래 코드에 맞게 유지해주세요.

# ... (이전 코드 생략: n_splits, skf, THRESHOLD, results, device, X_train_cnn, X_test_cnn, cnn_model, models 정의 부분)
# models 딕셔너리와 CNNClassifier, evaluate_model 함수는 정의되어 있다고 가정합니다.
THRESHOLD = 0.4

## 모델별 학습
for name, model in models.items():
    fold_metrics = []
    print(f"model name: {name}")

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        # Use .iloc for pandas DataFrames with integer indices
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # StandardScaler 적용
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_val_scaled = scaler.transform(X_val)

        # DataFrame 형태로 변환 (컬럼명 유지)
        X_tr_scaled = pd.DataFrame(X_tr_scaled, columns=X_tr.columns, index=X_tr.index)
        X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns, index=X_val.index)

        if name == '1D CNN PyTorch':
            # Convert to numpy arrays and reshape for CNN (스케일링된 데이터 사용)
            X_tr_cnn = X_tr_scaled.values.reshape((X_tr_scaled.shape[0], 1, X_tr_scaled.shape[1]))
            X_val_cnn = X_val_scaled.values.reshape((X_val_scaled.shape[0], 1, X_val_scaled.shape[1]))

            # Create fresh CNN instance for each fold
            cnn_model = CNNClassifier(
                input_channels=1,
                input_length=X_train.shape[1],
                epochs=20,
                batch_size=64,
                lr=0.0005,
                device=device
            )
            cnn_model.fit(X_tr_cnn, y_tr.values)

            # ------------------------------------------------------------------
            # 💡 훈련 스코어 출력 추가 (1D CNN PyTorch)
            y_tr_prob = cnn_model.predict_proba(X_tr_cnn)[:, 1]
            y_tr_pred = (y_tr_prob >= THRESHOLD).astype(int)
            train_result = evaluate_model(y_tr, y_tr_pred, y_tr_prob,
                                   algorithm_name=name,
                                   dataset_name=f"Fold {fold+1} Training")
            print(f"[{name}] Fold {fold+1} 훈련 스코어 (Accuracy/AUC): {train_result['Accuracy']:.4f} / {train_result['AUC']:.4f}")
            # ------------------------------------------------------------------

            y_prob = cnn_model.predict_proba(X_val_cnn)[:, 1]
            y_pred = (y_prob >= THRESHOLD).astype(int)

            # Clear GPU memory if using CUDA
            if device == 'cuda':
                torch.cuda.empty_cache()

        else:
            clf = clone(model)
            # 스케일링된 데이터로 학습
            clf.fit(X_tr_scaled, y_tr)

            # ------------------------------------------------------------------
            # 💡 훈련 스코어 출력 추가 (다른 모델들)
            if hasattr(clf, 'predict_proba'):
                y_tr_prob = clf.predict_proba(X_tr_scaled)[:, 1]
                y_tr_pred = (y_tr_prob >= THRESHOLD).astype(int)
            else:
                # decision_function 대체 로직
                y_tr_prob = clf.decision_function(X_tr_scaled)
                if y_tr_prob.max() != y_tr_prob.min():
                    y_tr_prob = (y_tr_prob - y_tr_prob.min()) / (y_tr_prob.max() - y_tr_prob.min())
                else:
                    y_tr_prob = np.full_like(y_tr_prob, 0.5)
                y_tr_pred = (y_tr_prob >= THRESHOLD).astype(int)

            train_result = evaluate_model(y_tr, y_tr_pred, y_tr_prob,
                                   algorithm_name=name,
                                   dataset_name=f"Fold {fold+1} Training")
            print(f"[{name}] Fold {fold+1} 훈련 스코어 (precision/recall/f1): {fold_result['Precision']:.4f} / {fold_result['Recall']:.4f} / {fold_result['F1-score']:.4f}")
            # ------------------------------------------------------------------

            # Handle probability prediction (스케일링된 데이터로 예측) - 검증 데이터
            if hasattr(clf, 'predict_proba'):
                y_prob = clf.predict_proba(X_val_scaled)[:, 1]
                y_pred = (y_prob >= THRESHOLD).astype(int)
            else:
                # Fallback to decision function for models like SVM
                y_prob = clf.decision_function(X_val_scaled)
                if y_prob.max() != y_prob.min():
                    y_prob = (y_prob - y_prob.min()) / (y_prob.max() - y_prob.min())
                else:
                    y_prob = np.full_like(y_prob, 0.5)
                # 커스텀 threshold로 예측 변환
                y_pred = (y_prob >= THRESHOLD).astype(int)

        # 검증 스코어 계산 및 저장 (기존 로직 유지)
        fold_result = evaluate_model(y_val, y_pred, y_prob,
                                   algorithm_name=name,
                                   dataset_name=f"Fold {fold+1} Validation")
        fold_metrics.append(fold_result)
        # 검증 스코어도 함께 출력 (선택 사항)
        print(f"[{name}] Fold {fold+1} 검증 스코어 (precision/recall/f1): {fold_result['Precision']:.4f} / {fold_result['Recall']:.4f} / {fold_result['F1-score']:.4f}")


    # Calculate average metrics across folds (기존 로직 유지)
    avg_result = {}
    numeric_keys = ['N', 'True Positive', 'True Negative', 'False Positive', 'False Negative',
                    'Precision', 'Recall', 'Specificity', 'F1-score', 'Accuracy',
                    'Balanced Accuracy', 'AUC']

    for key in numeric_keys:
        avg_result[key] = np.mean([res[key] for res in fold_metrics])

    avg_result['Algorithm'] = name
    results.append(avg_result)
    print("-" * 50)
    print(f"[{name}] 평균 검증 Accuracy: {avg_result['Accuracy']:.4f}, 평균 검증 AUC: {avg_result['AUC']:.4f}")
    print("-" * 50)
    # ------------------------------------------------------------------
    # 💡 2. 최종 모델 학습 및 X_test 평가 (진짜 테스트 결과)
    # ------------------------------------------------------------------

    # 1. 최종 모델 재훈련 (전체 X_train 데이터 사용)
    print(f"\n===== {name} 최종 모델 재훈련 및 X_test 평가 시작 =====")

    # 훈련 데이터 전체에 대한 스케일러 적용 (X_train과 X_test 모두에 적용)
    final_scaler = StandardScaler()
    X_train_final_scaled = final_scaler.fit_transform(X_train)
    # X_test_final_scaled: 훈련 데이터셋의 통계로 스케일링된 테스트 데이터
    X_test_final_scaled = final_scaler.transform(X_test)

    # DataFrame 형태로 변환
    X_train_final_scaled = pd.DataFrame(X_train_final_scaled, columns=X_train.columns)
    X_test_final_scaled = pd.DataFrame(X_test_final_scaled, columns=X_test.columns)

    if name == '1D CNN PyTorch':
        # CNN은 numpy 배열로 reshape 필요
        X_train_final_cnn = X_train_final_scaled.values.reshape((X_train_final_scaled.shape[0], 1, X_train_final_scaled.shape[1]))
        X_test_final_cnn = X_test_final_scaled.values.reshape((X_test_final_scaled.shape[0], 1, X_test_final_scaled.shape[1]))

        # 새로운 CNN 인스턴스를 생성하고 전체 훈련 데이터로 재훈련
        final_model = CNNClassifier(
            input_channels=1,
            input_length=X_train.shape[1],
            epochs=20, # K-Fold에서 사용한 동일 하이퍼파라미터 사용
            batch_size=64,
            lr=0.0005,
            device=device
        )
        final_model.fit(X_train_final_cnn, y_train.values)

        # Test Prediction
        y_prob_test = final_model.predict_proba(X_test_final_cnn)[:, 1]

        # GPU 메모리 정리
        if device == 'cuda':
            torch.cuda.empty_cache()

    else:
        # Scikit-learn 모델 재훈련
        final_model = clone(model)
        final_model.fit(X_train_final_scaled, y_train)

        # Test Prediction
        if hasattr(final_model, 'predict_proba'):
            y_prob_test = final_model.predict_proba(X_test_final_scaled)[:, 1]
        else:
            # predict_proba가 없는 모델을 위한 결정 함수 대체 로직
            y_prob_test = final_model.decision_function(X_test_final_scaled)
            if y_prob_test.max() != y_prob_test.min():
                y_prob_test = (y_prob_test - y_prob_test.min()) / (y_prob_test.max() - y_prob_test.min())
            else:
                y_prob_test = np.full_like(y_prob_test, 0.5)

    # 최종 예측 (THRESHOLD 적용)
    y_pred_test = (y_prob_test >= THRESHOLD).astype(int)

    # 2. X_test 성능 평가
    test_result = evaluate_model(y_test, y_pred_test, y_prob_test,
                                   algorithm_name=name,
                                   dataset_name="Final Test Set")

    # 3. 최종 테스트 결과 출력 및 results에 저장
    print("\n--------------------------------------------------")
    print(f"[{name}] 최종 테스트 스코어 (X_test)")
    print(f"Accuracy: {test_result['Accuracy']:.4f}")
    print(f"AUC: {test_result['AUC']:.4f}")
    print(f"F1-score: {test_result['F1-score']:.4f}")
    print("--------------------------------------------------")

    results.append(test_result)
# (모델 루프 종료)

model name: Logistic Regression
[Logistic Regression] Fold 1 훈련 스코어 (precision/recall/f1): 0.5744 / 0.8169 / 0.6745
[Logistic Regression] Fold 1 검증 스코어 (precision/recall/f1): 0.6355 / 0.7124 / 0.6718
[Logistic Regression] Fold 2 훈련 스코어 (precision/recall/f1): 0.6355 / 0.7124 / 0.6718
[Logistic Regression] Fold 2 검증 스코어 (precision/recall/f1): 0.6442 / 0.7194 / 0.6797
[Logistic Regression] Fold 3 훈련 스코어 (precision/recall/f1): 0.6442 / 0.7194 / 0.6797
[Logistic Regression] Fold 3 검증 스코어 (precision/recall/f1): 0.6474 / 0.7119 / 0.6782
--------------------------------------------------
[Logistic Regression] 평균 검증 Accuracy: 0.8299, 평균 검증 AUC: 0.8811
--------------------------------------------------

===== Logistic Regression 최종 모델 재훈련 및 X_test 평가 시작 =====

--------------------------------------------------
[Logistic Regression] 최종 테스트 스코어 (X_test)
Accuracy: 0.8285
AUC: 0.8807
F1-score: 0.6735
--------------------------------------------------
model name: Random Forest
[Random Forest] Fold 1 

KeyboardInterrupt: 

In [71]:
df_fold_results = pd.DataFrame(test_result).round(6)
display(df_fold_results)

,N,True Positive,True Negative,False Positive,False Negative,Precision,Recall,Specificity,F1-score,Accuracy,Balanced Accuracy,AUC,Algorithm
0,36983.0,7336.666667,22793.333333,4984.333333,1868.666667,0.595473,0.797002,0.820563,0.681652,0.814699,0.808782,0.881188,Logistic Regression
1,36983.0,8513.333333,17119.333333,10658.333333,692.000000,0.444101,0.924826,0.616299,0.600044,0.693093,0.770562,0.875646,Random Forest
2,36983.0,8289.666667,19279.000000,8498.666667,915.666667,0.493780,0.900528,0.694047,0.637823,0.745442,0.797288,0.883363,XGBoost
3,36983.0,9045.666667,7893.666667,19884.000000,159.666667,0.312705,0.982655,0.284173,0.474424,0.458030,0.633414,0.875407,LightGBM
4,36983.0,7607.666667,22026.666667,5751.000000,1597.666667,0.569496,0.826441,0.792963,0.674317,0.801296,0.809702,0.877884,CatBoost
5,36983.0,7044.333333,22807.333333,4970.333333,2161.000000,0.586275,0.765244,0.821067,0.663890,0.807173,0.793156,0.868858,MLP
6,36983.0,7241.000000,22686.666667,5091.000000,1964.333333,0.587238,0.786609,0.816723,0.672402,0.809228,0.801666,0.875427,1D CNN PyTorch
7,36983.0,7336.666667,22793.333333,4984.333333,1868.666667,0.595473,0.797002,0.820563,0.681652,0.814699,0.808782,0.881188,Logistic Regression
8,36983.0,8513.333333,17119.333333,10658.333333,692.000000,0.444101,0.924826,0.616299,0.600044,0.693093,0.770562,0.875646,Random Forest
9,36983.0,8289.666667,19279.000000,8498.666667,915.666667,0.493780,0.900528,0.694047,0.637823,0.745442,0.797288,0.883363,XGBoost
